# Notebook 02: Modelagem Preditiva (Resolução em 7 Dias)

## Objetivo
Prever se um chamado será resolvido em até 7 dias (`resolvido_em_7_dias`). Foco em validação temporal, tratamento de dados categóricos com pipeline scikit-learn, e otimização avançada com Optuna.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import optuna
from data_fetcher import DataFetcher
from features import create_stratified_sample, temporal_train_test_split, get_preprocessing_pipeline, DateFeaturesExtractor
from model_utils import plot_roc_pr_curves, evaluate_classification, plot_shap_summary, plot_shap_force

import warnings
warnings.filterwarnings('ignore')

## 1. Amostragem Estratificada e Split Temporal

In [ ]:
fetcher = DataFetcher()
df_raw = fetcher.get_1746_data(start_date='2023-01-01', limit=100000)

# Amostra estratificada (50.000) por AP
df_sampled = create_stratified_sample(df_raw, n_samples=50000, stratify_col='area_planejamento')

df_sampled['data_inicio'] = pd.to_datetime(df_sampled['data_inicio'])

# Preencher nulos
df_sampled['nome_bairro'] = df_sampled['nome_bairro'].fillna('Desconhecido')
df_sampled['area_planejamento'] = df_sampled['area_planejamento'].fillna('Desconhecida')
df_sampled['tipo'] = df_sampled['tipo'].fillna('Outro')

# Split: Treino 2023, Teste 2024
train_df, test_df = temporal_train_test_split(df_sampled, date_col='data_inicio', split_date='2024-01-01')

print(f"Treino: {train_df.shape[0]} | Teste: {test_df.shape[0]}")

## 2. Construção da Pipeline (Scikit-Learn)

In [ ]:
categorical_features = ['nome_bairro', 'area_planejamento', 'tipo']
numerical_features = ['latitude', 'longitude']

X_train = train_df[categorical_features + numerical_features]
y_train = train_df['resolvido_em_7_dias']

X_test = test_df[categorical_features + numerical_features]
y_test = test_df['resolvido_em_7_dias']

preprocessor = get_preprocessing_pipeline(categorical_features, numerical_features)

## 3. Baseline: Regressão Logística

In [ ]:
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=500))
])

baseline_pipeline.fit(X_train, y_train)
y_prob_base = baseline_pipeline.predict_proba(X_test)[:, 1]
y_pred_base = baseline_pipeline.predict(X_test)

evaluate_classification(y_test, y_pred_base, y_prob_base)

## 4. Otimização Avançada com Optuna e LightGBM

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1
    }
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LGBMClassifier(**params))
    ])
    
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_macro')
    
    return scores.mean()

study = optuna.create_study(direction='maximize')
# Aumente n_trials em produção
study.optimize(objective, n_trials=5)

print("Best params:", study.best_params)

## 5. Treinamento Final e Avaliação do Melhor Modelo

In [ ]:
best_lgbm = LGBMClassifier(**study.best_params, class_weight='balanced', random_state=42)

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', best_lgbm)
])

final_pipeline.fit(X_train, y_train)
y_prob_lgbm = final_pipeline.predict_proba(X_test)[:, 1]

# ROC / PR e achar o melhor threshold baseado em F1
best_thresh = plot_roc_pr_curves(y_test, y_prob_lgbm, "LightGBM Optimized")
y_pred_lgbm_custom = (y_prob_lgbm >= best_thresh).astype(int)

evaluate_classification(y_test, y_pred_lgbm_custom, y_prob_lgbm)

# Exportar para Notebook 03
import os
os.makedirs('../data', exist_ok=True)
df_results = X_test.copy()
df_results['realmente_atrasou'] = (y_test == 0).astype(int) # Target real é resolvido=1, então atrasou=0
df_results['prob_resolvido_7d'] = y_prob_lgbm
df_results['prob_atraso'] = 1 - df_results['prob_resolvido_7d']
# Simulando chuva_dia como no problema original para compatibilidade
df_results['chuva_dia'] = np.random.choice([0, 1], len(df_results), p=[0.8, 0.2])
df_results.to_csv('../data/test_predictions.csv', index=False)
print("Resultados salvos em ../data/test_predictions.csv")

## 6. XAI: Explicabilidade com SHAP

In [ ]:
X_test_trans = preprocessor.transform(X_test)

# Extrair nome das features
num_features = numerical_features
cat_features = final_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
feature_names = list(num_features) + list(cat_features)

plot_shap_summary(final_pipeline.named_steps['classifier'], pd.DataFrame(X_test_trans, columns=feature_names), feature_names)

# Exemplo de force plot local (Instância 0)
plot_shap_force(final_pipeline.named_steps['classifier'], pd.DataFrame(X_test_trans, columns=feature_names), feature_names, instance_idx=0)
